In [0]:
from pyspark.sql import functions as f
file_name= ['drivers.json','driver_flat.parquet']
# Due to the nested column in the json source files laoding from here
path_flat ="abfss://bronze@stdevelopmentgani.dfs.core.windows.net/formula1fullloadlanding/drivers/drivers_flat"
path = "abfss://bronze@stdevelopmentgani.dfs.core.windows.net/formula1fullloadlanding/drivers"

df = spark.read.format('json')\
    .load(f"{path}/{file_name[0]}")

df_flat = df.withColumn('familyName',f.col('name.familyName'))\
    .withColumn('givenName',f.col('name.givenName'))\
    .withColumn('ingestion_timestamp',f.current_timestamp())\
    .withColumn('source_file_name', f.col('_metadata.file_path'))\
    .drop(f.col('name'))

df_flat.write.format("parquet").mode('overwrite').save(path_flat)